In [1]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

X_train = []
y_train = []

for _, row in train.iterrows():

    prompt = row["prompt"]

    for option in ["A", "B", "C", "D", "E"]:

        X_train.append(
            prompt + " " + str(row[option])
        )

        y_train.append(
            1 if row["answer"] == option else 0
        )

print("Training samples:", len(X_train))

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_vec = vectorizer.fit_transform(X_train)

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_vec, y_train)

print("Training complete")

Training samples: 10000
Training complete


In [5]:
predictions = []

for _, row in test.iterrows():

    scores = {}

    for option in ["A", "B", "C", "D", "E"]:

        text = row["prompt"] + " " + str(row[option])

        vec = vectorizer.transform([text])

        score = model.predict_proba(vec)[0][1]

        scores[option] = score

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    predictions.append(
        " ".join(ranked[:3])
    )

In [6]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

submission.head()

,ID,Prediction
0,1,A E C
1,2,B E A
2,3,B E D
3,4,E C A
4,5,C A B


In [7]:
!ls /kaggle/working

submission.csv
